# SaRoHead - LoRA fine-tuning of RoLLMs

One (model, seed) unit takes 1-3 h; split `SEEDS` across sessions.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/sarohead")
REPO = Path("/content/SaRoHead")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/GRAI-UNSTPB/SaRoHead.git", str(REPO)], check=True)

RESULTS = DRIVE / "results" / "predictions"
RESULTS.mkdir(parents=True, exist_ok=True)
os.environ["SAROHEAD_RESULTS"] = str(RESULTS)
sys.path.insert(0, str(REPO))
from analysis_scripts.prediction_io import prediction_filename

SEEDS = [13, 42, 123, 2024, 7]
CATEGORIES = ["social", "politic", "sport"]


def run(cmd, cwd):
    print(">>", " ".join(map(str, cmd)), flush=True)
    subprocess.run([str(c) for c in cmd], cwd=str(cwd), check=True)


def done(method, model, variant, category, seed):
    return (RESULTS / method / prediction_filename(method, model, variant, category, seed, "random", "test")).exists()


In [ ]:
!pip install -q peft trl==0.22.2 datasets transformers accelerate
!pip uninstall -y -q torchao torchaudio

import torch

MODELS = ["llama3", "llama2", "gemma", "mistral"]
PAPER = {"llama2": "RoLlama2", "llama3": "RoLlama3", "gemma": "RoGemma", "mistral": "RoMistral"}
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
BATCH_SIZE, GRAD_ACCUM = (16, 2) if VRAM_GB >= 60 else (8, 4)


In [ ]:
for seed in SEEDS:
    for model in MODELS:
        cats = [c for c in CATEGORIES if not done("lora", PAPER[model], "lora", c, seed)]
        if not cats:
            continue
        run([sys.executable, "new_lora_finetune.py", "--model_type", model, "--lora_r", 8, "--lora_alpha", 8,
             "--lora_dropout", 0.05, "--epochs", 5, "--lr", "1e-3", "--batch_size", BATCH_SIZE,
             "--grad_accum", GRAD_ACCUM, "--warmup_ratio", 0.2, "--seed", seed, "--categories", *cats],
            REPO / "LLM_scripts")
        run([sys.executable, "test_lora.py", "--model_name", model, "--category", *cats, "--seed", seed],
            REPO / "LLM_scripts")
        for c in cats:
            shutil.rmtree(REPO / "LLM_scripts" / f"check_main_model_{model}_{c}_random_seed{seed}", ignore_errors=True)
            shutil.rmtree(REPO / "LLM_scripts" / f"main_model_{model}_{c}_random_seed{seed}", ignore_errors=True)


In [ ]:
files = sorted(RESULTS.rglob("*.csv"))
print(len(files), "prediction files in", RESULTS)
